In [1]:
import os
import yaml
import os
import yaml
import warnings; warnings.filterwarnings('ignore')

from dotenv import load_dotenv
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [2]:
load_dotenv(dotenv_path="../.env") 

with open("../config/config.yaml", "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

env_var_name = config['api_keys']['google_gemini_env_name']
gemini_api_key = os.getenv(env_var_name)

if gemini_api_key:
    os.environ["GOOGLE_API_KEY"] = gemini_api_key
    print("✅ Gemini API 키가 성공적으로 로드되었습니다.")
else:
    print("❌ API 키를 찾을 수 없습니다. .env 파일을 확인해주세요.")

✅ Gemini API 키가 성공적으로 로드되었습니다.


In [3]:
pdf_path = "../data/raw/samsung_report.pdf" # 노트북 폴더 기준 상대 경로
try:
    loader = PyMuPDFLoader(pdf_path)
    documents = loader.load()
    print(f"✅ PDF 로드 완료: 총 {len(documents)} 페이지")
except Exception as e:
    print(f"❌ PDF를 찾을 수 없습니다. 경로를 확인해주세요: {e}")

✅ PDF 로드 완료: 총 527 페이지


In [4]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = text_splitter.split_documents(documents)
print(f"✅ 문서 분할 완료: 총 {len(splits)} 개의 텍스트 조각(Chunk) 생성")

✅ 문서 분할 완료: 총 1802 개의 텍스트 조각(Chunk) 생성


In [5]:
print("⏳ 임베딩 모델을 불러오는 중...")
embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")

⏳ 임베딩 모델을 불러오는 중...


C:\Users\kyle0\AppData\Local\Temp\ipykernel_5584\1192879546.py:2: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="jhgan/ko-sroberta-multitask")
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 28440.32it/s]
RobertaModel LOAD REPORT from: jhgan/ko-sroberta-multitask
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
vectorstore = Chroma.from_documents(documents=splits, embedding=embedding_model)
print("✅ Vector DB 구축 완료!")

✅ Vector DB 구축 완료!


In [7]:
print("\n" + "="*50)
print("🧐 AI에게 어려운 질문을 던져봅시다 (표 데이터 검색 테스트)")

query = "2024년 삼성전자의 당기순이익은 얼마인가요?"
print(f"질문: {query}\n")

# 검색기 셋팅 (가장 유사한 문서 3개 가져오기)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("🧠 Gemini에게 검색된 문서를 주고 답변을 생성하게 합니다...\n")


🧐 AI에게 어려운 질문을 던져봅시다 (표 데이터 검색 테스트)
질문: 2024년 삼성전자의 당기순이익은 얼마인가요?

🧠 Gemini에게 검색된 문서를 주고 답변을 생성하게 합니다...



In [8]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

In [9]:
prompt = ChatPromptTemplate.from_template("""
당신은 꼼꼼한 금융 데이터 분석가입니다.
아래 제공된 [검색된 문서]만을 바탕으로 사용자의 [질문]에 답변해주세요.
표 데이터가 깨져서 숫자가 헷갈린다면, 유추하지 말고 "문서에서 명확한 수치를 찾기 어렵습니다."라고 답변하세요.

[검색된 문서]
{context}

[질문]
{input}
""")

In [10]:
document_chain = create_stuff_documents_chain(llm, prompt)
retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [ ]:
response = retrieval_chain.invoke({"input": query})

print("=" * 50)
print("🤖 최종 AI 답변:")
print(response["answer"])
print("=" * 50)

🤖 최종 AI 답변:
제공된 문서에서 2024년 삼성전자의 당기순이익에 대한 명확한 수치를 찾기 어렵습니다. 문서에는 2024년 연결기준 매출, 영업이익, 부채비율, 자기자본비율, ROE 등의 정보만 제시되어 있습니다.
